# House Prices — Advanced Regression Techniques
**Goal:** Predict the sale price of houses in Ames, Iowa.

**Metric:** RMSE on log(SalePrice)

**Pipeline:**
1. EDA
2. Data Cleaning & Smart NA Filling
3. Feature Engineering
4. Encoding
5. Modeling (LightGBM with Cross-Validation)
6. Submission

## 1. Imports & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# Save IDs for submission
train_ids = train['Id']
test_ids = test['Id']

print('Train shape:', train.shape)
print('Test shape:', test.shape)

## 2. EDA

In [ ]:
train.head()

In [ ]:
# SalePrice distribution — it is right-skewed, so we log-transform it
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train['SalePrice'], bins=50)
axes[0].set_title('SalePrice (original)')
axes[1].hist(np.log1p(train['SalePrice']), bins=50)
axes[1].set_title('SalePrice (log-transformed)')
plt.tight_layout()
plt.show()

In [ ]:
# Missing values in train
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
# Correlation of numerical features with SalePrice
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
corr = train[num_cols].corr()['SalePrice'].sort_values(ascending=False)
print(corr.head(15))

## 3. Data Cleaning & Smart NA Filling

**Key insight:** Many NAs in this dataset don't mean "missing" — they mean "None".
For example, `PoolQC = NA` means no pool, `GarageType = NA` means no garage.
We fill these with 'None' or 0 instead of median/mode.

In [ ]:
# Log-transform SalePrice — makes the target normally distributed
# which is what regression models prefer
y = np.log1p(train['SalePrice'])

# Combine train and test for consistent preprocessing
train = train.drop(['Id', 'SalePrice'], axis=1)
test = test.drop(['Id'], axis=1)
n_train = len(train)
df = pd.concat([train, test], axis=0).reset_index(drop=True)

print('Combined shape:', df.shape)

In [ ]:
# Categorical columns where NA means 'None' (no feature present)
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in none_cols:
    df[col] = df[col].fillna('None')

# Numerical columns where NA means 0 (no feature present)
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]
for col in zero_cols:
    df[col] = df[col].fillna(0)

# LotFrontage — fill with median of neighborhood (houses in same area are similar)
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Remaining NAs — fill categorical with mode, numerical with median
for col in df.columns:
    if df[col].isna().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print('Remaining NAs:', df.isna().sum().sum())

## 4. Feature Engineering

In [ ]:
# Total square footage — one of the strongest predictors
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# House age and remodel age
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']

# Total bathrooms
df['TotalBath'] = (df['FullBath'] + df['BsmtFullBath'] +
                   0.5 * df['HalfBath'] + 0.5 * df['BsmtHalfBath'])

# Total porch area
df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                      df['3SsnPorch'] + df['ScreenPorch'])

# Has pool, has garage, has basement flags
df['HasPool'] = (df['PoolArea'] > 0).astype(int)
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)

print('Features after engineering:', df.shape[1])

## 5. Encoding

In [ ]:
# Label encode all categorical columns
# LightGBM handles label encoded categoricals well
cat_cols = df.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print('Categorical columns encoded:', len(cat_cols))
print('Final shape:', df.shape)

In [ ]:
# Split back into train and test
X = df[:n_train]
X_test_final = df[n_train:]

print('X shape:', X.shape)
print('X_test_final shape:', X_test_final.shape)

## 6. Modeling

Using LightGBM with 5-fold cross-validation for a reliable performance estimate.

In [ ]:
# LightGBM with cross-validation
model = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print('CV RMSE scores:', -cv_scores)
print('Mean CV RMSE:', (-cv_scores).mean())
print('Std CV RMSE:', (-cv_scores).std())

## 7. Submission

In [ ]:
# Train final model on all training data
model.fit(X, y)

# Predict on test set — remember to reverse the log transform with expm1
log_predictions = model.predict(X_test_final)
final_predictions = np.expm1(log_predictions)

# Create submission file
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})
submission.to_csv('submission.csv', index=False)
print('Submission saved!')
print(submission.head())